# Starling 6U — attitude regulation (PD + bias MEKF)

This notebook supports **Section 3 — Attitude Regulation**: hold a target attitude using a nonlinear **PD** controller and the course-style **MEKF with gyro bias** in the loop, with **environmental disturbances** from [environmental_perturbations.jl](environmental_perturbations.jl) (same context as [report4.ipynb](report4.ipynb)) and **RWP050** limits (6 mN·m torque, 50 mN·m·s internal wheel momentum \(\rho\) per axis).

**Implementation** lives in [attitude_control_starling_impl.jl](attitude_control_starling_impl.jl) (reusable, testable). Run this notebook from the `code` directory so `include` paths resolve.

**Deliverables**
- Nonlinear **PD** regulation using \(\phi=\log q_e\) and bias-corrected gyro rate.
- **Monte Carlo** initial attitudes up to **±90°**, **multi-orbit** LEO (GG + drag), **gyro / vector noise**, **RMS pointing** over a steady window.

**Quaternion / measurement convention** Hamilton \(L(q)\) as in MEKF notebooks; body observation of inertial unit vector \(\mathbf b = Q(q)^\top \mathbf r\) (matches [mekf-simple.ipynb](mekf-simple.ipynb)).

**Methodology** is spelled out in the next markdown section. The **demo code** cell prints the PD gains and sensor/bias-noise settings used by the closed-loop simulation.

## Controller design methodology (what this notebook implements)

### Objective
Regulate body attitude **\(q\)** (body-to-inertial quaternion) to a constant command **\(q_d\)**, using noisy gyro and vector observations processed by the **course MEKF**. The truth plant runs in closed loop with RWP050-class torque and internal wheel-momentum limits (`τ_rw_max`, `rho_rw_max`).

### Lecture-aligned notation
- **Total angular momentum:** \(H = J\omega + \rho\).
- **Internal wheel momentum:** \(\rho\), with \(\dot\rho=-\tau_{cmd}\) under the reaction-torque convention used here.
- **Gyro model:** \(\tilde\omega_k=\omega_k+\beta_k+v_k\), with \(\beta_k\) modeled as a random walk and estimated by the MEKF.

### Feedback signal
- **Attitude:** \(\hat q\) from the MEKF multiplicative update.
- **Rate:** bias-corrected gyro rate, \(\omega_{ctrl}=\tilde\omega_{LPF}-\hat\beta\), when the course MEKF path is active.

### Error attitude (shortest rotation)
Target-relative error quaternion **\(q_e = q_d^{-1} \otimes \hat q\)** (implemented as `quat_err` in the impl). If **\(q_{e,0}<0\)**, replace **\(q_e \leftarrow -q_e\)** so the rotation is **\(\leq 180^\circ\)**.

### Controller — nonlinear PD
- \(\phi = \log(q_e)\) (principal rotation vector; same spirit as `cmg.ipynb` / Lecture 18).
- \(\tau_{cmd} = -k_p \phi - k_d \omega_{ctrl}\).
- No integral or switched linear branch is used in the controller scheme.

### MEKF (course path, two inertial directions)
The nominal path is the course MEKF: state \([q;\beta]\), 6×6 covariance on \([\phi;\delta\beta]\), gyro-bias correction in prediction, and vector updates using sun + nadir body observations. The older attitude-only 3×3 MEKF remains in the implementation for comparison/debugging only.

### Environmental torque
**Gravity gradient + atmospheric drag** via `environmental_wrench` (SRP off for LEO choice, consistent with report4). Orbit is **circular**; **\(\nu\)** advances each step.

### Where the math becomes code
| Piece | Julia |
|-------|--------|
| Plant + RW saturation | `dynamics_rhs`, `rk4_step` |
| PD controller | `PDGains`, `torque_pd` |
| Bias MEKF | `ProfessorCourseMEKF.mekf_predict`, `ProfessorCourseMEKF.mekf_update` |
| Closed loop | `simulate_closed_loop` |

The next **code** cell runs a PD+MEKF demo and prints the simulation parameters you can paste into a report.

In [ ]:
using LinearAlgebra
using StaticArrays
using Random
using Statistics
using PythonPlot

# Restart kernel and run this cell once if you see "conflicting import" after re-including modules.
include("environmental_perturbations.jl")
include("attitude_control_starling_impl.jl")

Random.seed!(42)

## Tune gains and run a single PD demo

Adjust `g_pd` if you need a different torque vs pointing trade-off. The next cell prints the PD gains, sensor noise settings, and runs one closed-loop PD+MEKF demonstration using the course bias MEKF (`professor_mekf=true`) and internal wheel momentum `rho`.

In [ ]:
dt_ctrl = 0.25

g_pd = PDGains(2.5, 3.5)

sigma_gyro = 0.002
sigma_vec = 0.01
sigma_bias_walk = 1e-5
V_mekf = Diagonal(fill(max((sigma_gyro * dt_ctrl)^2, 1e-10), 3))

println("======== Controller design record (Starling nominal) ========")
println("Sample period dt = ", dt_ctrl, " s")
println("PD gains: kp = ", g_pd.kp, ", kd = ", g_pd.kd)
println("Sensor noise (MEKF): sigma_gyro = ", sigma_gyro, ", sigma_vec = ", sigma_vec, ", sigma_bias_walk = ", sigma_bias_walk)
println("Nominal estimator: course MEKF with gyro bias; wheel momentum variable: rho")
println("============================================================")

qd = Vector(quat_normalize(@SVector [1.0, 0.0, 0.0, 0.0]))
q0 = Vector(random_quat_max_angle(deg2rad(85)))
omega0 = zeros(3)
rho0 = zeros(3)

sol_pd = simulate_closed_loop(
    Tfinal=6000.0,
    dt=dt_ctrl,
    q0=q0,
    omega0=omega0,
    rho0=rho0,
    qd=qd,
    g=g_pd,
    sigma_gyro=sigma_gyro,
    sigma_vec=sigma_vec,
    V_mekf=V_mekf,
    rng=MersenneTwister(7),
    control_gyro_lpf_tau=0.5,
    professor_mekf=true,
    sigma_bias_walk=sigma_bias_walk,
)

println("Demo RMS pointing (deg, last 75%): ", rms_steady(collect(sol_pd.ang_hist); skip_frac=0.25))

figure(figsize=(10, 4))
subplot(1, 2, 1)
plot(collect(sol_pd.t) ./ 60, sol_pd.ang_hist, color="C0")
xlabel("Time [min]")
ylabel("Pointing error [deg]")
title("Principal angle (truth vs q_d)")
grid(true)
subplot(1, 2, 2)
plot(collect(sol_pd.t) ./ 60, norm.(eachcol(sol_pd.tau_hist)), color="C1")
xlabel("Time [min]")
ylabel("||tau|| [N*m]")
title("Commanded torque magnitude (PD)")
grid(true)
tight_layout()

In [ ]:
# PD-only controller scheme.
# Use the single PD demo cell above for the nominal closed-loop run.

### Course MEKF (`mekf_model_notation.jl`) vs simplified 3-state MEKF

The **Professor / course MEKF** is implemented in [`mekf_model_notation.jl`](mekf_model_notation.jl): **7 states** (\(q\) + gyro bias \(\beta\)), **6×6** error covariance, vector updates with \(\mathbf y_{\mathrm{pred}} = Q(q)^\top \mathbf r_N\) (same \(\mathbf b=Q^\top\mathbf r\) convention as lecture notes). It is wrapped in `ProfessorCourseMEKF` and selected with **`professor_mekf=true`** in `simulate_closed_loop`. This is now the **nominal reported path**. The gyro rate used for control is \(\omega_{\mathrm{gyro,LPF}}-\hat\beta\).

The simplified comparison MEKF in the impl keeps a **3×3** covariance on \(\phi\) only and does **not** estimate \(\hat\beta\). It remains useful for debugging, but it is not the lecture-aligned nominal estimator.

**Reading the plotted “errors”.** **`ang_hist`** is **pointing**: principal angle **truth vs commanded** \(q_d\) (performance of the regulator). **`ang_est_hist`** is **estimation**: **truth vs \(\hat q\)**. Lab-style filter-only RMS is closer to \(\mathrm{RMS}(\mathtt{ang\_est\_hist})\) than to \(\mathrm{RMS}(\mathtt{ang\_hist})\) when you compare notebooks.

In [ ]:
qd_id = Vector(quat_normalize(@SVector [1.0, 0.0, 0.0, 0.0]))
q0_cmp = Vector(random_quat_max_angle(deg2rad(55)))
omega0_cmp = zeros(3)
rho0_cmp = zeros(3)
T_cmp = 400.0

sol_simple = simulate_closed_loop(
    Tfinal=T_cmp, dt=dt_ctrl, q0=q0_cmp, omega0=omega0_cmp, rho0=rho0_cmp, qd=qd_id,
    g=g_pd, sigma_gyro=sigma_gyro, sigma_vec=sigma_vec, V_mekf=V_mekf, rng=MersenneTwister(2026),
    control_gyro_lpf_tau=0.5,
    professor_mekf=false,
    sigma_bias_walk=sigma_bias_walk,
)
sol_prof = simulate_closed_loop(
    Tfinal=T_cmp, dt=dt_ctrl, q0=q0_cmp, omega0=omega0_cmp, rho0=rho0_cmp, qd=qd_id,
    g=g_pd, sigma_gyro=sigma_gyro, sigma_vec=sigma_vec, V_mekf=V_mekf, rng=MersenneTwister(2026),
    control_gyro_lpf_tau=0.5,
    professor_mekf=true,
    sigma_bias_walk=sigma_bias_walk,
)

println("=== Same ICs/noise draws (RNG 2026), T=$(T_cmp)s ===")
println("Simple MEKF — RMS pointing vs q_d (skip 25%): ", round(rms_steady(collect(sol_simple.ang_hist); skip_frac=0.25), digits=4), " deg")
println("Simple MEKF — RMS truth vs qhat           : ", round(rms_steady(collect(sol_simple.ang_est_hist); skip_frac=0.25), digits=4), " deg")
println("Professor MEKF — RMS pointing vs q_d      : ", round(rms_steady(collect(sol_prof.ang_hist); skip_frac=0.25), digits=4), " deg")
println("Professor MEKF — RMS truth vs qhat        : ", round(rms_steady(collect(sol_prof.ang_est_hist); skip_frac=0.25), digits=4), " deg")

## Monte Carlo (±90°), multi-orbit, PD only

`T_mc` spans **15 circular orbits** at the nominal altitude. Reduce `n_orbits` for a quicker local run.

In [ ]:
T_orbit_s = 2pi / mean_motion_rad_s
n_orbits = 15
T_mc = n_orbits * T_orbit_s
Nseeds = 12

rms_pd = Float64[]

for s in 1:Nseeds
    rng = MersenneTwister(100 + s)
    q0 = Vector(random_quat_max_angle(deg2rad(90)))
    omega0 = 0.02 .* randn(rng, 3)
    rho0 = 0.01 .* rho_rw_max .* randn(rng, 3)

    sol_pd = simulate_closed_loop(
        Tfinal=T_mc, dt=dt_ctrl, q0=q0, omega0=omega0, rho0=rho0, qd=qd,
        g=g_pd, sigma_gyro=sigma_gyro, sigma_vec=sigma_vec, V_mekf=V_mekf, rng=rng,
        control_gyro_lpf_tau=0.5,
        professor_mekf=true,
        sigma_bias_walk=sigma_bias_walk,
    )
    push!(rms_pd, rms_steady(collect(sol_pd.ang_hist); skip_frac=0.2))
end

println("Orbit period (s) ≈ ", round(T_orbit_s, digits=1), ", MC horizon (s) = ", round(T_mc, digits=1))
println("RMS pointing [deg] — PD + MEKF: mean=", round(mean(rms_pd), digits=4), ", max=", round(maximum(rms_pd), digits=4))

figure(figsize=(7, 4))
x = 1:Nseeds
plot(x, rms_pd, "o-", label="PD + MEKF")
xlabel("Seed")
ylabel("RMS pointing error [deg]")
title("Steady-window RMS ($(n_orbits) orbits, GG+drag)")
legend()
grid(true)
tight_layout()

## Write-up notes

- **PD** handles large \(\|q_e\|\) without relying on a single linearization or switched controller.
- **Wheel momentum notation:** use \(\rho\) for internal reaction-wheel momentum. Total angular momentum is \(H=J\omega+\rho\).
- **MEKF** nominally uses gyro propagation with bias, two inertial directions (sun + nadir), and a 6×6 covariance on \([\phi;\delta\beta]\). The 3×3 attitude-only MEKF is only a debugging/comparison option.
- **Gyro model:** \(\tilde\omega=\omega+\beta+v\). With `professor_mekf=true`, the torque law uses \(\omega_{ctrl}=\tilde\omega_{LPF}-\hat\beta\).
- **Why torque can look noisy:** With `control_gyro_lpf_tau=0`, the D term sees fresh discrete gyro white noise at every control step. **`control_gyro_lpf_tau > 0`** applies a first-order low-pass to the rate used for torque only; the MEKF still uses the raw gyro sample for propagation.
- Optional **`control_from_truth=true`** in `simulate_closed_loop` isolates plant vs filter for debugging.